In [ ]:
%env PORTKEY_API_KEY=...

In [1]:
import os
from pydantic_ai import Agent
from pydantic_ai.capabilities import MCP
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

from IPython.display import display, Markdown

## Let us go one step higher in the abstraction by using Agents and letting them decide which tools to call rather than specifying them manually

In [2]:
agent = Agent(
    name="hpc-docs-agent",
    model=OpenAIResponsesModel(
        model_name="@vertexai/gemini-3.7-flash",
        provider=OpenAIProvider(
            base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
            api_key=os.getenv("PORTKEY_API_KEY"),
        ),
    ),
    instructions="Be concise and answer questions from retreived knowledge with tools.",
    capabilities=[
        MCP(
            url="https://mcp-gateway.apps.cloud.rt.nyu.edu/rts-docs-algolia-public/mcp",
            id="rts-docs-algolia-mcp",
            headers={
                "x-portkey-api-key": os.getenv("PORTKEY_API_KEY"),
            },
        ),
    ],
)

In [3]:
result = await agent.run("Login to HPC cluster from off campus") # await is added because the agent is run asynchronously

In [4]:
display(Markdown(result.output))

To log into the NYU HPC (Torch) cluster from off-campus, follow these steps:

---

### Step 1: Connect to the NYU VPN
Access from outside the campus network requires an active NYU VPN connection.
* Connect using your configured **NYU VPN client** before attempting to log in.

---

### Step 2: Connect to the Cluster

You can connect via **SSH (Command Line)** or **Open OnDemand (Web Interface)**:

#### Option A: Command Line (SSH)
1. Open your terminal (macOS/Linux Terminal, Windows PowerShell, WSL, or Git Bash/PuTTY).
2. Run:
   ```bash
   ssh <NetID>@login.torch.hpc.nyu.edu
   ```
3. Complete the two-factor authentication prompt:
   * Open the provided URL ([https://microsoft.com/devicelogin](https://microsoft.com/devicelogin)).
   * Enter the one-time code shown in the terminal.
   * Log in with `<NetID>@nyu.edu` and approve the MFA request via Duo.
   * Return to your terminal and press **Enter**.

---

#### Option B: Open OnDemand (Web Browser)
1. While connected to the VPN, open your browser and go to:  
   **[https://ood.torch.hpc.nyu.edu](https://ood.torch.hpc.nyu.edu)**
2. Log in with your NYU credentials.
3. Access the command line via **Clusters > Torch Shell Access**, or launch interactive apps (e.g., Jupyter, RStudio, Desktop).

## Let's see how we can evaluate the performance of the Agent on this task by varying the LLM used. 
## Here we check if the output from the Agent contains the specific string `login.torch.hpc.nyu.edu` to ensure that the model did not hallucinate a new login address:

In [5]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Contains

# Create a dataset with test cases
dataset = Dataset(
    name='login-to-hpc',
    cases=[
        Case(
            name="use-gemini-3.7-flash",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-3.7-flash"
            },
        ),
        Case(
            name="use-gemini-2.5-flash-lite",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-2.5-flash-lite"
            },
        ),
    ],
    evaluators=[
        Contains(value='login.torch.hpc.nyu.edu', case_sensitive=True),
    ],
)

async def agent_task(inputs: dict) -> str:
    with agent.override(model=
                        OpenAIResponsesModel(
                            model_name=inputs["model"],
                            provider=OpenAIProvider(
                                base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
                                api_key=os.getenv("PORTKEY_API_KEY"),
                                ),
                        )
                       ):
        result = await agent.run(user_prompt=inputs["query"])
        return result.output


# Run the evaluation
report = await dataset.evaluate(agent_task)

# Print the results
report.print()

Output()

           Evaluation Summary: agent_task            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Case ID                   ┃ Assertions ┃ Duration ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━┩
│ use-gemini-3.7-flash      │ ✔          │    17.3s │
├───────────────────────────┼────────────┼──────────┤
│ use-gemini-2.5-flash-lite │ ✗          │     4.0s │
├───────────────────────────┼────────────┼──────────┤
│ Averages                  │ 50.0% ✔    │    10.7s │
└───────────────────────────┴────────────┴──────────┘

## Why did the case with `gemini-2.5-flash-lite` fail? Let's check the output from that run:

In [6]:
display(Markdown(report.cases[1].output))

I found documentation on how to log into the burst login node using SSH. You can log in via the VPN, and the documentation also mentions adding a line to your SSH config file.

Would you like me to provide the specific instructions?

## The older, less capable model misunderstood the prompt and answered the question for a different HPC cluster (Cloud bursting). Explore how you may prevent this?